In [1]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
import sys
import os
sys.path.append("benchmarks")

In [3]:
os.getcwd()

'/home/bkantz/cgv/hereditary/Dense-KG-Vector-DB'

In [4]:
from utils.datasets.utils import parse_nt_to_generator, save_from_generator
from utils.datasets.dbpedia import DBPedia
from pathlib import Path

In [5]:
dataset=DBPedia(
    base_dir=Path("benchmarks/data/dbpedia/")
)

In [6]:
dataset.full_ttl_file, dataset.full_ttl_file.exists()

(PosixPath('benchmarks/data/dbpedia/dbpedia_complete.nt'), False)

In [13]:
from utils.dbs.base_db import BaseDB
from utils.dbs.qlever_native import QleverDBNative
from utils.dbs.qlever import QleverDB

db = QleverDBNative(
    dataset=dataset,
    id="dbpedia-encoded",
    name="QleverDB-DBPedia",
    base_dir=dataset.base_dir / "index-encoded",
    use_encoded_ttl=True,
)
# db = QleverDB(
#     dataset=dataset,
#     id="dbpedia-encoded",
#     name="QleverDB-DBPedia",
#     base_dir=dataset.base_dir / "index-encoded",
#     use_encoded_ttl=False,
# )
db.id = "dbpedia-encoded-tidx"
db.db_dir = dataset.base_dir / "index-encoded"
db.base_dir = dataset.base_dir / "index-encoded"

2026-11-19 11:04:12,744 - WARNING - Killing any existing process using port 26044 before starting the server
2026-11-19 11:04:12,750 - ERROR - Command failed with return code 1
2026-11-19 11:04:12,751 - INFO - Initialized QLeverDBNative with id=dbpedia-encoded, port_id=26044, dataset=DBPedia, name=QleverDB-DBPedia, use_encoded_ttl=True, endpoint=http://localhost:26044/dbpedia-encoded-with-tidx/sparql


In [14]:
db.db_dir, db.base_dir

(PosixPath('benchmarks/data/dbpedia/index-encoded'),
 PosixPath('benchmarks/data/dbpedia/index-encoded'))

In [15]:
db.log_file

PosixPath('logs/QleverDBNative.log')

In [19]:
with db:
    res=db.query("""SELECT (COUNT(?s) as ?count)
    WHERE {
        ?s dbo:thumbnail_embedding ?thumbnail .
    }""")
res

2026-11-19 11:10:58,419 - INFO - Setting up QLeverDBNative db_dir=benchmarks/data/dbpedia/index-encoded, base_dir=benchmarks/data/dbpedia/index-encoded
2026-11-19 11:10:58,422 - INFO - Logging QLever setup to benchmarks/data/dbpedia/index-encoded/db/dbpedia-encoded-with-tidx/dbpedia-encoded-with-tidx_run.log
2026-11-19 11:10:58,423 - INFO - Loading dataset into QLever server from benchmarks/data/dbpedia/dbpedia_complete.nt
2026-11-19 11:10:58,423 - WARNING - DB directory benchmarks/data/dbpedia/index-encoded already exists!
2026-11-19 11:10:58,424 - INFO - Stopping server!


2026-11-19 11:10:58,455 - ERROR - Command failed with return code 1
2026-11-19 11:10:58,455 - INFO - Starting QLever server on port 26044
2026-11-19 11:10:58,456 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 26044 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-11-19 11:10:58,457 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26044/dbpedia-encoded-with-tidx/sparql)
2026-11-19 11:10:59,471 - INFO - Server is up and responding to queries
2026-11-19 11:10:59,532 - INFO - Stopping server!


 2508964

26044/tcp:          


,count
0,2329464


In [26]:
with db:
    res =db.query("""PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
            PREFIX dta: <https://w3id.org/rdf-tensor/aggregates#>
            PREFIX dbr: <http://dbpedia.org/resource/>
            PREFIX dbo: <http://dbpedia.org/ontology/>
            SELECT DISTINCT ?clsB (dta:avg(?vectorB) AS ?vectorBmean) WHERE {
                    ?instB a ?clsB ;
                    dbo:thumbnail_embedding ?vectorB .
                    FILTER (?clsB = dbo:Ship)
            } GROUP BY ?clsB
            LIMIT 10""")
res

2026-11-19 11:16:37,767 - INFO - Setting up QLeverDBNative db_dir=benchmarks/data/dbpedia/index-encoded, base_dir=benchmarks/data/dbpedia/index-encoded
2026-11-19 11:16:37,768 - INFO - Logging QLever setup to benchmarks/data/dbpedia/index-encoded/db/dbpedia-encoded-with-tidx/dbpedia-encoded-with-tidx_run.log
2026-11-19 11:16:37,769 - INFO - Loading dataset into QLever server from benchmarks/data/dbpedia/dbpedia_complete.nt
2026-11-19 11:16:37,770 - WARNING - DB directory benchmarks/data/dbpedia/index-encoded already exists!
2026-11-19 11:16:37,770 - INFO - Stopping server!


2026-11-19 11:16:37,807 - ERROR - Command failed with return code 1
2026-11-19 11:16:37,808 - INFO - Starting QLever server on port 26044
2026-11-19 11:16:37,808 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 26044 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-11-19 11:16:37,809 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26044/dbpedia-encoded-with-tidx/sparql)
2026-11-19 11:16:38,825 - INFO - Server is up and responding to queries
2026-11-19 11:16:38,989 - INFO - Stopping server!


 2530394

26044/tcp:          


,clsB,vectorBmean
0,dbo:Ship,"{""data"":[0.025510853,-0.17345604,-0.14521839,0..."


In [ ]:
with db:
    res =db.query("""PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
            PREFIX dta: <https://w3id.org/rdf-tensor/aggregates#>
            PREFIX dbr: <http://dbpedia.org/resource/>
            PREFIX dbo: <http://dbpedia.org/ontology/>
            SELECT DISTINCT ?clsB (dta:avg(?vectorB) AS ?vectorBmean) WHERE {
                    ?instB a ?clsB ;
                    dbo:thumbnail_embedding ?vectorB .
                    FILTER (?clsB = dbo:Ship)
            } GROUP BY ?clsB
            LIMIT 10""")
res

2026-11-19 11:16:37,767 - INFO - Setting up QLeverDBNative db_dir=benchmarks/data/dbpedia/index-encoded, base_dir=benchmarks/data/dbpedia/index-encoded
2026-11-19 11:16:37,768 - INFO - Logging QLever setup to benchmarks/data/dbpedia/index-encoded/db/dbpedia-encoded-with-tidx/dbpedia-encoded-with-tidx_run.log
2026-11-19 11:16:37,769 - INFO - Loading dataset into QLever server from benchmarks/data/dbpedia/dbpedia_complete.nt
2026-11-19 11:16:37,770 - WARNING - DB directory benchmarks/data/dbpedia/index-encoded already exists!
2026-11-19 11:16:37,770 - INFO - Stopping server!


2026-11-19 11:16:37,807 - ERROR - Command failed with return code 1
2026-11-19 11:16:37,808 - INFO - Starting QLever server on port 26044
2026-11-19 11:16:37,808 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 26044 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-11-19 11:16:37,809 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26044/dbpedia-encoded-with-tidx/sparql)
2026-11-19 11:16:38,825 - INFO - Server is up and responding to queries
2026-11-19 11:16:38,989 - INFO - Stopping server!


 2530394

26044/tcp:          


,clsB,vectorBmean
0,dbo:Ship,"{""data"":[0.025510853,-0.17345604,-0.14521839,0..."


In [63]:
with db:
    res =db.query("""PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
            PREFIX dta: <https://w3id.org/rdf-tensor/aggregates#>
            PREFIX dbr: <http://dbpedia.org/resource/>
            PREFIX dbo: <http://dbpedia.org/ontology/>
            SELECT ?instA ?clsB ?vectorA ?vectorBmean ?dist 
            WHERE {
                ?instA a ?clsB .
                ?instA dbo:thumbnail_embedding ?vectorA .
                {
                    SELECT ?clsB (dta:avg(?vectorB) AS ?vectorBmean) WHERE {
                         ?instB a ?clsB ;
                         dbo:thumbnail_embedding ?vectorB .
                        FILTER (?clsB = dbo:Ship)
                    } GROUP BY ?clsB
                }
                BIND(dtf:cosineSimilarity(?vectorA, ?vectorBmean) AS ?dist)
            } GROUP BY ?instA ?clsB ?vectorA ?vectorBmean ?dist
            ORDER BY DESC(?dist)
            LIMIT 10""")
res

2026-11-19 11:52:32,313 - INFO - Setting up QLeverDBNative db_dir=benchmarks/data/dbpedia/index-encoded, base_dir=benchmarks/data/dbpedia/index-encoded
2026-11-19 11:52:32,314 - INFO - Logging QLever setup to benchmarks/data/dbpedia/index-encoded/db/dbpedia-encoded-with-tidx/dbpedia-encoded-with-tidx_run.log
2026-11-19 11:52:32,314 - INFO - Loading dataset into QLever server from benchmarks/data/dbpedia/dbpedia_complete.nt
2026-11-19 11:52:32,314 - WARNING - DB directory benchmarks/data/dbpedia/index-encoded already exists!
2026-11-19 11:52:32,315 - INFO - Stopping server!


2026-11-19 11:52:32,339 - ERROR - Command failed with return code 1
2026-11-19 11:52:32,339 - INFO - Starting QLever server on port 26044
2026-11-19 11:52:32,339 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 26044 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-11-19 11:52:32,340 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26044/dbpedia-encoded-with-tidx/sparql)
2026-11-19 11:52:33,348 - INFO - Server is up and responding to queries
2026-11-19 11:52:33,616 - INFO - Stopping server!


 2609171

26044/tcp:          


,instA,clsB,vectorA,vectorBmean,dist
0,dbr:USS_Alden,dbo:Ship,"{""data"": [-0.16596846282482147, -0.30987873673...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",0.9358044266701
1,dbr:BYMS-class_minesweeper,dbo:Ship,"{""data"": [-0.005602780729532242, -0.4926316440...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",0.9351377487183
2,dbr:HMS_Cygnet_(H83),dbo:Ship,"{""data"": [-0.11827000975608826, -0.59536445140...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",0.9343208670616
3,dbr:USCGC_Campbell_(WPG-32),dbo:Ship,"{""data"": [-0.20242707431316376, -0.43034261465...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",0.9306849241257
4,dbr:HMS_Ascension,dbo:Ship,"{""data"": [0.08045569062232971, -0.353212654590...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",0.9304448366165
5,dbr:HMS_Locust_(T28),dbo:Ship,"{""data"": [0.0601324737071991, -0.4549732208251...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",0.9303362369537
6,dbr:USS_Lionfish,dbo:Ship,"{""data"": [0.15550416707992554, -0.285803496837...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",0.9300966858864
7,dbr:MS_Bantam_(1939),dbo:Ship,"{""data"": [-0.2853374481201172, -0.377013415098...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",0.9298396706581
8,dbr:USS_Oxford_(APA-189),dbo:Ship,"{""data"": [0.11346741020679474, -0.416568875312...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",0.9297803044319
9,dbr:USCGC_Courier_(WAGR-410),dbo:Ship,"{""data"": [-0.19793111085891724, -0.41754969954...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",0.9297365546227


In [ ]:
with db:
    res =db.query("""PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
            PREFIX dta: <https://w3id.org/rdf-tensor/aggregates#>
            PREFIX dbr: <http://dbpedia.org/resource/>
            PREFIX dbo: <http://dbpedia.org/ontology/>
            PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
            SELECT ?instA ?clsB ?vectorA ?vectorBmean ?dist 
            WHERE {                
                {
                    SELECT ?clsB (dta:avg(?vectorB) AS ?vectorBmean) WHERE {
                         ?instB a ?clsB ;
                         dbo:thumbnail_embedding ?vectorB .
                        FILTER (?clsB = dbo:Ship)
                    } GROUP BY ?clsB
                }     
                SERVICE tensorIndex: {
                    _:config tensorIndex:numNN 10 ;
                    tensorIndex:left ?vectorBmean ;
                    tensorIndex:bindDistance ?dist ;
                    tensorIndex:payload ?instA, ?clsB, ?vectorBmean ;
                    tensorIndex:searchK 16 ;
                    tensorIndex:kIVF 128 ;
                    tensorIndex:experimentalRightCacheName "easy_group_index_dbpedia" ;
                    tensorIndex:right ?vectorA ;
                    tensorIndex:algorithm tensorIndex:ivf ;
                    tensorIndex:distance tensorIndex:dot .
                    {
                        ?instA a ?clsB .       
                        ?instA dbo:thumbnail_embedding ?vectorA .   
                        FILTER (?clsB = dbo:Ship)  
                    }
                }  
            }
            LIMIT 10""")
res

2026-11-19 11:52:40,899 - INFO - Setting up QLeverDBNative db_dir=benchmarks/data/dbpedia/index-encoded, base_dir=benchmarks/data/dbpedia/index-encoded
2026-11-19 11:52:40,900 - INFO - Logging QLever setup to benchmarks/data/dbpedia/index-encoded/db/dbpedia-encoded-with-tidx/dbpedia-encoded-with-tidx_run.log
2026-11-19 11:52:40,900 - INFO - Loading dataset into QLever server from benchmarks/data/dbpedia/dbpedia_complete.nt
2026-11-19 11:52:40,900 - WARNING - DB directory benchmarks/data/dbpedia/index-encoded already exists!
2026-11-19 11:52:40,901 - INFO - Stopping server!
2026-11-19 11:52:40,926 - ERROR - Command failed with return code 1
2026-11-19 11:52:40,926 - INFO - Starting QLever server on port 26044
2026-11-19 11:52:40,926 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 26044 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-11-19 11:52:40,927 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://loca

2026-11-19 11:52:41,936 - INFO - Server is up and responding to queries
2026-11-19 11:52:42,901 - INFO - Stopping server!


 2609582

26044/tcp:          


,instA,clsB,vectorA,vectorBmean,dist
0,dbr:HNLMS_O_24,dbo:Ship,"{""data"": [-0.028958380222320557, -0.1799555420...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",89.24034881592
1,dbr:USS_Golet,dbo:Ship,"{""data"": [-0.10507935285568237, -0.19751948118...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",86.33767700195
2,dbr:USS_Turner_(DD-648),dbo:Ship,"{""data"": [0.22411513328552246, -0.282439738512...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",85.87467956543
3,dbr:USS_Raton,dbo:Ship,"{""data"": [0.3259657323360443, -0.3372522592544...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",85.4831237793
4,dbr:HMAS_Australia_(D84),dbo:Ship,"{""data"": [0.09161266684532166, -0.025912404060...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",85.1349029541
5,dbr:USS_York_County_(LST-1175),dbo:Ship,"{""data"": [0.0698281079530716, -0.3883688747882...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",85.00476837158
6,dbr:Indefatigable-class_battlecruiser,dbo:Ship,"{""data"": [0.08531245589256287, -0.173149257898...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",84.95151519775
7,dbr:Japanese_submarine_I-51,dbo:Ship,"{""data"": [0.00572165846824646, -0.194036930799...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",84.88075256348
8,dbr:HNoMS_Ula_(1943),dbo:Ship,"{""data"": [0.07497040927410126, -0.438187301158...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",84.7926940918
9,dbr:Requin-class_submarine,dbo:Ship,"{""data"": [0.3334200382232666, 0.20390337705612...","{""data"":[0.025510853,-0.17345604,-0.14521839,0...",84.60521697998


In [62]:
with db:
    res =db.query("""PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
            PREFIX dta: <https://w3id.org/rdf-tensor/aggregates#>
            PREFIX dbr: <http://dbpedia.org/resource/>
            PREFIX dbo: <http://dbpedia.org/ontology/>
            PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>       
            SELECT ?clsB( COUNT(?clsB) AS ?count) WHERE {
                    ?instB a ?clsB .
            } GROUP BY ?clsB """)
res

2026-11-19 11:51:47,169 - INFO - Setting up QLeverDBNative db_dir=benchmarks/data/dbpedia/index-encoded, base_dir=benchmarks/data/dbpedia/index-encoded
2026-11-19 11:51:47,170 - INFO - Logging QLever setup to benchmarks/data/dbpedia/index-encoded/db/dbpedia-encoded-with-tidx/dbpedia-encoded-with-tidx_run.log
2026-11-19 11:51:47,170 - INFO - Loading dataset into QLever server from benchmarks/data/dbpedia/dbpedia_complete.nt
2026-11-19 11:51:47,170 - WARNING - DB directory benchmarks/data/dbpedia/index-encoded already exists!
2026-11-19 11:51:47,170 - INFO - Stopping server!
2026-11-19 11:51:47,195 - ERROR - Command failed with return code 1
2026-11-19 11:51:47,195 - INFO - Starting QLever server on port 26044
2026-11-19 11:51:47,195 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 26044 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-11-19 11:51:47,196 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://loca

2026-11-19 11:51:48,204 - INFO - Server is up and responding to queries
2026-11-19 11:51:48,326 - INFO - Stopping server!


 2607436

26044/tcp:          


,clsB,count
0,dbo:Academic,10663
1,dbo:AcademicConference,99
2,dbo:AcademicJournal,9364
3,dbo:Activity,17971
4,dbo:Actor,4901
...,...,...
786,http://www.wikidata.org/entity/Q9609,239
787,http://www.wikidata.org/entity/Q9620,324
788,http://www.wikidata.org/entity/Q9655,330
789,http://xmlns.com/foaf/0.1/Document,16916804


In [65]:
with db:
    res =db.query("""PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
            PREFIX dta: <https://w3id.org/rdf-tensor/aggregates#>
            PREFIX dbr: <http://dbpedia.org/resource/>
            PREFIX dbo: <http://dbpedia.org/ontology/>
            PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>  
            SELECT ?instA ?clsB ?vectorA ?vectorBmean ?dist 
            WHERE {
                ?instA a ?clsB .
                ?instA dbo:thumbnail_embedding ?vectorA .
                {
                    SELECT ?clsB (dta:avg(?vectorB) AS ?vectorBmean) WHERE {
                         ?instB a ?clsB ;
                         dbo:thumbnail_embedding ?vectorB .
                    } GROUP BY ?clsB
                }
                BIND(dtf:cosineSimilarity(?vectorA, ?vectorBmean) AS ?dist)
            } GROUP BY ?instA ?clsB ?vectorA ?vectorBmean ?dist
            ORDER BY DESC(?dist)
            LIMIT 10""")
res

2026-11-19 11:53:08,363 - INFO - Setting up QLeverDBNative db_dir=benchmarks/data/dbpedia/index-encoded, base_dir=benchmarks/data/dbpedia/index-encoded
2026-11-19 11:53:08,363 - INFO - Logging QLever setup to benchmarks/data/dbpedia/index-encoded/db/dbpedia-encoded-with-tidx/dbpedia-encoded-with-tidx_run.log
2026-11-19 11:53:08,363 - INFO - Loading dataset into QLever server from benchmarks/data/dbpedia/dbpedia_complete.nt
2026-11-19 11:53:08,364 - WARNING - DB directory benchmarks/data/dbpedia/index-encoded already exists!
2026-11-19 11:53:08,364 - INFO - Stopping server!
2026-11-19 11:53:08,389 - ERROR - Command failed with return code 1
2026-11-19 11:53:08,389 - INFO - Starting QLever server on port 26044
2026-11-19 11:53:08,389 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 26044 -k 0 -m 16G --tensor-search-max-num-threads 4


2026-11-19 11:53:08,390 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26044/dbpedia-encoded-with-tidx/sparql)
2026-11-19 11:53:09,399 - INFO - Server is up and responding to queries
2026-11-19 11:53:39,410 - ERROR - Query failed: (429, 'HTTP Error 429: Too Many Requests', None)
2026-11-19 11:53:39,411 - INFO - Stopping server!


 2610518

26044/tcp:          


TimeoutError: 

In [74]:
with db:
    res =db.query("""PREFIX rdf: <http://www.w3.org/2000/01/rdf-schema#>
            PREFIX dtf: <https://w3id.org/rdf-tensor/functions#>
            PREFIX dta: <https://w3id.org/rdf-tensor/aggregates#>
            PREFIX dbr: <http://dbpedia.org/resource/>
            PREFIX dbo: <http://dbpedia.org/ontology/>
            PREFIX tensorIndex: <https://qlever.cs.uni-freiburg.de/tensorIndex/>
            SELECT DISTINCT ?instA ?clsB ?vectorA ?vectorBmean ?dist 
            WHERE {                
                {
                    SELECT ?clsB (dta:avg(?vectorB) AS ?vectorBmean) WHERE {
                         ?instB a ?clsB ;
                         dbo:thumbnail_embedding ?vectorB .
                    } GROUP BY ?clsB
                }     
                SERVICE tensorIndex: {
                    _:config tensorIndex:numNN 1 ;
                    tensorIndex:left ?vectorBmean ;
                    tensorIndex:bindDistance ?dist ;
                    tensorIndex:payload ?instA, ?clsB, ?vectorBmean ;
                    tensorIndex:searchK 1 ;
                    tensorIndex:kIVF 128 ;
                    tensorIndex:experimentalRightCacheName "hard_group_index_dbpedia" ;
                    tensorIndex:right ?vectorA ;
                    tensorIndex:algorithm tensorIndex:ivf ;
                    tensorIndex:distance tensorIndex:dot .
                    {
                        ?instA a ?clsB .       
                        ?instA dbo:thumbnail_embedding ?vectorA .   
                    }
                }  
            }
            GROUP BY ?instA ?clsB ?vectorA ?vectorBmean ?dist
            ORDER BY DESC(?dist)
            LIMIT 10 """)
res

2026-11-19 12:01:36,348 - INFO - Setting up QLeverDBNative db_dir=benchmarks/data/dbpedia/index-encoded, base_dir=benchmarks/data/dbpedia/index-encoded
2026-11-19 12:01:36,349 - INFO - Logging QLever setup to benchmarks/data/dbpedia/index-encoded/db/dbpedia-encoded-with-tidx/dbpedia-encoded-with-tidx_run.log
2026-11-19 12:01:36,349 - INFO - Loading dataset into QLever server from benchmarks/data/dbpedia/dbpedia_complete.nt
2026-11-19 12:01:36,350 - WARNING - DB directory benchmarks/data/dbpedia/index-encoded already exists!
2026-11-19 12:01:36,350 - INFO - Stopping server!


2026-11-19 12:01:36,376 - ERROR - Command failed with return code 1
2026-11-19 12:01:36,376 - INFO - Starting QLever server on port 26044
2026-11-19 12:01:36,376 - INFO - Running command: qlever-server -i dbpedia-encoded-tidx --port 26044 -k 0 -m 16G --tensor-search-max-num-threads 4
2026-11-19 12:01:36,377 - INFO - Waiting for server to start..., got error: <urlopen error [Errno 111] Connection refused> (http://localhost:26044/dbpedia-encoded-with-tidx/sparql)
2026-11-19 12:01:37,386 - INFO - Server is up and responding to queries
2026-11-19 12:01:52,126 - INFO - Stopping server!


 2629269

26044/tcp:          


,instA,clsB,vectorA,vectorBmean,dist
0,dbr:Walchia,dbo:Conifer,"{""data"": [-0.20487765967845917, 0.262808740139...","{""data"":[-0.20487766,0.26280874,-0.2595622,-0....",128.0424194336
1,dbr:1985_Hopmann,dbo:CelestialBody,"{""data"": [0.32152819633483887, 0.2336313426494...","{""data"":[0.36054498,0.15991442,-0.09124743,-0....",117.2889709473
2,dbr:Beighton_Cup,dbo:Agent,"{""data"": [0.16601458191871643, 0.0372639149427...","{""data"":[0.16601458,0.037263915,-0.23329867,-0...",112.0975418091
3,dbr:Kaiten,dbo:Device,"{""data"": [-0.10355229675769806, -0.13713888823...","{""data"":[-0.071987525,-0.11454872,-0.2924546,0...",108.1427078247
4,dbr:Angostura_Diversion_Dam,dbo:ArchitecturalStructure,"{""data"": [-0.40367263555526733, 0.258369892835...","{""data"":[-0.40367264,0.2583699,0.09689723,0.27...",107.947265625
5,dbr:Professional_Fighters_League,dbo:Agent,"{""data"": [0.5620298385620117, 0.04845375195145...","{""data"":[0.56202984,0.048453752,-0.004743889,-...",106.283782959
6,dbr:Jamesoniella,dbo:Eukaryote,"{""data"": [-0.19049254059791565, 0.096104949712...","{""data"":[-0.17316028,0.064043894,-0.16744737,0...",105.823425293
7,dbr:Mesityl_oxide,dbo:ChemicalCompound,"{""data"": [0.13079069554805756, -0.015390537679...","{""data"":[0.07237041,-0.11825936,-0.2330035,0.0...",105.5582427979
8,dbr:Houghton_Hall,http://www.w3.org/2003/01/geo/wgs84_pos#Spatia...,"{""data"": [0.11666251718997955, 0.0128811597824...","{""data"":[0.20027772,-0.016733816,-0.22261158,0...",105.3859863281
9,dbr:CorA_metal_ion_transporter,dbo:Biomolecule,"{""data"": [0.3884204030036926, -0.1030780598521...","{""data"":[0.35061264,-0.13335763,0.10400853,-0....",105.1381607056
